In [25]:
from utils.application import *#
from utils.dynamicRieszFunctions import estimateDynamicRiesz_all
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.estimateDiDLinear import estimateDiDLinear
import torch
import pandas as pd
import time
from torch.distributions import Normal
from utils.dgp import DiD_DGP
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

## Settings

lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' :0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'poly_degree' : 0,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' :1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}
rf_f_settings = {
    'poly_degree' : 1, # 1 or 2?
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' :1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}





In [ ]:
seed = 123
folds = 5

data_application = application_data()

In [17]:
data_dict = data_application.get_data(2003, 2004, baseline_2001=True)
X1 = data_dict['X1']
X2 = data_dict['X2']
Y1 = data_dict['Y1']
Y2 = data_dict['Y2']
Z = data_dict['Z']
D = data_dict['D']

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4', '2001_emp0A01_BS', '2001_pop', '2001_annual_avg_pay', '2001_lpop', '2001_lavg_pay']


In [18]:
ATT, STD, *_ = estimateDynamicRiesz(Y1, Y2, D, Z, X1, X2, folds,
                                                                     method_a = "RF", rf_a_settings=rf_a_settings,
                                                                        method_f = "RF", rf_f_settings = rf_f_settings, lasso_a_settings= lasso_a_settings, lasso_f_settings=lasso_f_settings)
print("ATT: " ,ATT, f"({STD})")

ATT:  tensor(-0.0080) (0.6670489311218262)


In [ ]:
ATTs, STDs = [], []

for i in tqdm(range(20)):
   data_dict = data_application.get_data(2003, 2004, baseline_2001=False)
   X1 = data_dict['X1']
   X2 = data_dict['X2']
   Y1 = data_dict['Y1']
   Y2 = data_dict['Y2']
   Z = data_dict['Z']
   D = data_dict['D']
   ATT, STD, *_ = estimateDynamicRiesz(Y1, Y2, D, Z, X1, X2, folds,
                                                                        method_a = "RF", rf_a_settings=rf_a_settings,
                                                                           method_f = "RF", rf_f_settings = rf_f_settings)
   ATTs.append(ATT)
   STDs.append(STD)


  0%|          | 0/20 [00:00<?, ?it/s]

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4']


  5%|▌         | 1/20 [00:53<17:04, 53.90s/it]

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4']


 10%|█         | 2/20 [01:46<15:52, 52.92s/it]

...

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4']


 95%|█████████▌| 19/20 [16:36<00:52, 52.52s/it]

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4']


100%|██████████| 20/20 [17:28<00:00, 52.45s/it]

ATT:  tensor(-0.0211) (0.7607274055480957)


In [26]:
np.mean(ATTs)

-0.020265434

In [ ]:
ATTsBL, STDsBL = [], []

for i in tqdm(range(20)):
   data_dict = data_application.get_data(2003, 2004, baseline_2001=True)
   X1 = data_dict['X1']
   X2 = data_dict['X2']
   Y1 = data_dict['Y1']
   Y2 = data_dict['Y2']
   Z = data_dict['Z']
   D = data_dict['D']
   ATT, STD, *_ = estimateDynamicRiesz(Y1, Y2, D, Z, X1, X2, folds,
                                                                        method_a = "RF", rf_a_settings=rf_a_settings,
                                                                           method_f = "RF", rf_f_settings = rf_f_settings)
   ATTsBL.append(ATT)
   STDsBL.append(STD)


  0%|          | 0/20 [00:00<?, ?it/s]

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4', '2001_emp0A01_BS', '2001_pop', '2001_annual_avg_pay', '2001_lpop', '2001_lavg_pay']


  5%|▌         | 1/20 [02:21<44:48, 141.51s/it]

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4', '2001_emp0A01_BS', '2001_pop', '2001_annual_avg_pay', '2001_lpop', '2001_lavg_pay']


 10%|█         | 2/20 [04:44<42:41, 142.32s/it]

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4', '2001_emp0A01_BS', '2001_pop', '2001_annual_avg_pay', '2001_lpop', '2001_lavg_pay']


 15%|█▌        | 3/20 [07:07<40:29, 142.89s/it]

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4', '2001_emp0A01_BS', '2001_pop', '2001_annual_avg_pay', '2001_lpop', '2001_lavg_pay']


 20%|██        | 4/20 [09:31<38:09, 143.10s/it]

Changing covariates:  ['emp0A01_BS', 'pop', 'annual_avg_pay', 'lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4', '2001_emp0A01_BS', '2001_pop', '2001_annual_avg_pay', '2001_lpop', '2001_lavg_pay']
